# Cerberus Compliance — KYB Quickstart

**KYB** (Know Your Business) is the regulated practice of verifying a
company's legal identity, ownership structure, and regulatory standing
before onboarding it as a customer or counterparty. In Chile, this means
checking public registries, sanctions lists (OFAC, EU, UN, CMF-ONU),
essential facts published under NCG 30, and the regulatory frameworks the
entity is subject to (Ley 21.521 RPSF, Ley 21.719, NCG 514, etc.).

This notebook gets you from an API key to useful KYB data in about five
minutes. Each step has a short explanation of *why* before the code, so
it's designed to be read top-to-bottom the first time through.

> 📘 Full docs at [developers.cerberus.cl](https://developers.cerberus.cl) (dev portal live post-P7).

## Prerequisites

- **Python 3.10+** (the SDK uses modern type hints and `match` statements).
- The SDK itself: `pip install cerberus-compliance`.
- A Cerberus API key. The client reads it from the `CERBERUS_API_KEY`
  environment variable by default — export it before launching Jupyter,
  or pass `api_key=` explicitly.
- Optional: `pip install rich` for the pretty-printed summary at the end.
  The notebook degrades gracefully to `print()` when `rich` isn't installed.

In [ ]:
# Uncomment the next line on a fresh kernel to install the SDK and the
# optional `rich` dependency. It's commented out so re-running the whole
# notebook doesn't mutate your environment.
# !pip install cerberus-compliance rich

## Step 1 — Create a client

`CerberusClient` is the single entry point for the SDK. By default it
reads `CERBERUS_API_KEY` from the environment, but every knob is
overrideable:

- `api_key=` — override the env var (useful for notebooks with multiple keys).
- `base_url=` — point at staging (`https://api-staging.cerberus.cl/v1`) or a
  local mock server.
- `timeout=`, `max_retries=`, `backoff_factor=` — tune the retry policy
  (exponential backoff with jitter; 429 and 5xx are retried by default).

In [ ]:
import os

from cerberus_compliance import CerberusClient

client = CerberusClient(
    api_key=os.environ.get("CERBERUS_API_KEY", "ck_demo_replace_me"),
    base_url=os.environ.get("CERBERUS_BASE_URL", "https://compliance.cerberus.cl/v1"),
)
client

## Step 2 — Pick a RUT

The Chilean **RUT** (Rol Único Tributario) is the national tax ID. The
canonical format is `NN.NNN.NNN-D`, where `D` is a verifier digit (`0`–`9`
or `K`). The SDK accepts both the dotted form and the raw digits.

We'll use **Falabella Retail S.A.** (`76.086.428-5`) as a public example —
it's a listed issuer with plenty of material events and a well-populated
regulatory profile, so every step below will return non-trivial data.

In [ ]:
rut = "76.086.428-5"  # Falabella Retail S.A. — public example

## Step 3 — Resolve the entity

Entities are queried by RUT through `GET /entities?rut=...`. The response
is a standard list envelope:

```json
{"data": [ {"id": "...", "legal_name": "...", ...} ], "next": null}
```

For this quickstart we use the low-level `client._request(...)` escape
hatch, which is available today. Once the resources PR lands, the more
ergonomic `client.entities.get(rut=rut)` will replace every `_request`
call in this notebook.

In [ ]:
# TODO(#P4-B-merge): switch to client.entities.get(rut=rut) once feat/resources-1 lands.
response = client._request("GET", "/entities", params={"rut": rut, "limit": 1})
entities = response.get("data", [])
entity = entities[0] if entities else None
entity

## Step 4 — List recent essential facts

Under Chilean securities law (**NCG 30**, issued by the CMF), listed
issuers must disclose *hechos esenciales* — material facts that a
reasonable investor would consider important. These include M&A
announcements, board changes, covenant breaches, dividend decisions, and
material litigation.

For ongoing KYB monitoring, this feed is the single best signal that
something about a counterparty has changed since you onboarded them.

In [ ]:
# TODO(#P4-B-merge): switch to client.entities.material_events(id=entity["id"], limit=5).
events_resp = client._request(
    "GET",
    f"/entities/{entity['id']}/material_events",
    params={"limit": 5},
)
events = events_resp.get("data", [])
events

## Step 5 — Active sanctions

Cerberus aggregates sanctions lists from the major international bodies
and the Chilean regulator:

- **OFAC** (U.S. Treasury Specially Designated Nationals)
- **EU** consolidated financial sanctions
- **UN** Security Council Consolidated List
- **CMF-ONU** (Chilean CMF's mirror of the UN list, binding locally)

Passing `active=true` filters out historical and lifted sanctions so you
only see entries that still apply. For audit and historical exposure
reviews, drop the flag to see the full timeline.

In [ ]:
# TODO(#P4-B-merge): switch to client.entities.sanctions(id=entity["id"], active=True).
sanctions_resp = client._request(
    "GET",
    f"/entities/{entity['id']}/sanctions",
    params={"active": "true"},
)
sanctions = sanctions_resp.get("data", [])
sanctions

## Step 6 — Regulatory profile

The `regulations` endpoint surfaces the frameworks an entity is subject
to. For a Chilean financial counterparty that typically includes:

- **Ley 21.521 (Fintec/RPSF)** — Registro de Prestadores de Servicios
  Financieros, the fintech perimeter law.
- **Ley 21.719 (Datos Personales)** — the 2024 data protection overhaul.
- **NCG 514** — operational risk and cyber resilience for supervised entities.

Each item includes the obligations triggered, the supervising body, and
any active enforcement actions. It's the fastest way to scope a
compliance due-diligence review.

In [ ]:
# TODO(#P4-B-merge): switch to client.entities.regulations(id=entity["id"]).
regs_resp = client._request("GET", f"/entities/{entity['id']}/regulations")
regulations = regs_resp.get("data", [])
regulations

## Step 7 — Pretty-print with `rich` (optional)

[`rich`](https://github.com/Textualize/rich) isn't a hard dependency of
the SDK, but it makes ad-hoc exploration much nicer. The cell below
attempts the import and falls back to plain `print()` when `rich` is
absent, so it works either way.

In [ ]:
try:
    from rich.console import Console
    from rich.table import Table

    console = Console()
    table = Table(title=f"KYB profile — {entity['legal_name']} ({rut})")
    table.add_column("Metric")
    table.add_column("Value")
    table.add_row("Material events (last 5)", str(len(events)))
    table.add_row("Active sanctions", str(len(sanctions)))
    table.add_row("Regulatory frameworks", str(len(regulations)))
    console.print(table)
except ImportError:
    print(f"KYB profile — {entity['legal_name']} ({rut})")
    print(f"  material events (last 5): {len(events)}")
    print(f"  active sanctions: {len(sanctions)}")
    print(f"  regulatory frameworks: {len(regulations)}")

## Clean up

`CerberusClient` is a context manager. This notebook created one *outside*
a `with` block on purpose — it lets you re-run individual cells without
rebuilding the HTTP session each time, which is great for experimentation.

In production, prefer the context-manager form so the underlying
connection pool is always released cleanly:

```python
with CerberusClient() as client:
    response = client._request("GET", "/entities", params={"rut": rut})
```

When used outside a `with` block, call `.close()` explicitly before the
kernel shuts down.

In [ ]:
client.close()

## Next steps

Other examples in the repository:

- [`examples/kyb_quickstart.py`](../kyb_quickstart.py) — the CLI version
  of this notebook, good for scripting and CI smoke tests.
- [`examples/monitor_portfolio.py`](../monitor_portfolio.py) — a polling
  workflow that watches a list of RUTs for new material events.
- [`examples/webhook_handler.py`](../webhook_handler.py) — a push-based
  FastAPI receiver with signature verification.

Docs (written alongside this notebook):

- [`docs/quickstart.md`](../../docs/quickstart.md)
- [`docs/auth.md`](../../docs/auth.md)
- [`docs/errors.md`](../../docs/errors.md)
- [`docs/pagination.md`](../../docs/pagination.md)

And the dev portal placeholder (live post-P7):
[developers.cerberus.cl](https://developers.cerberus.cl).